In [1]:
# =========================
# Cell 1 - Cài thư viện nếu thiếu
# =========================
!pip install mp-api pymatgen ase pandas tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 kB 4.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.7/155.7 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.1/829.1 kB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 118.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.6/41.6 MB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 121.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 127.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.5/140.5 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 122.3 MB/s eta 0:00:00
   ━━━━━━━

In [2]:
# =========================
# Cell 2 - Import
# =========================
import os
import re
import pandas as pd
from tqdm import tqdm

from pymatgen.core import Composition
from mp_api.client import MPRester
from ase.io import read

In [3]:
# =========================
# Cell 3 - Hàm xử lý công thức
# =========================
def is_stoichiometric_formula(formula):
    """
    Lọc công thức stoichiometry:
    - Không chứa ngoặc, dấu %, +, -, /, x, y
    - Parse được bằng pymatgen
    - Có tổng hệ số nguyên hoặc gần nguyên
    """
    try:
        f = str(formula).strip()

        if any(ch in f for ch in ["(", ")", "[", "]", "%", "+", "-", "/", "x", "y"]):
            return False

        comp = Composition(f)

        # kiểm tra có parse được nguyên tố thật
        if len(comp.elements) == 0:
            return False

        # reduced formula chuẩn
        reduced = comp.reduced_formula

        # loại công thức quá lỗi
        if reduced is None or reduced == "":
            return False

        return True

    except Exception:
        return False


def normalize_formula(formula):
    """
    Đưa công thức về dạng reduced formula để query Materials Project.
    Ví dụ: Bi2Te3 -> Bi2Te3
    """
    return Composition(str(formula)).reduced_formula

In [4]:
# =========================
# Cell 4 - Hàm tìm cấu trúc từ Materials Project
# =========================
def get_lowest_energy_structure_from_mp(formula, api_key):
    """
    Input:
        formula: công thức hóa học, ví dụ 'Bi2Te3'
        api_key: Materials Project API key

    Output:
        dict chứa mp-id, formula, structure, cif_text
        hoặc None nếu không tìm thấy
    """

    formula_query = normalize_formula(formula)

    with MPRester(api_key) as mpr:
        docs = mpr.materials.summary.search(
            formula=formula_query,
            fields=[
                "material_id",
                "formula_pretty",
                "formation_energy_per_atom",
                "structure"
            ]
        )

    if len(docs) == 0:
        return None

    # chọn cấu trúc có formation energy thấp nhất
    best_doc = min(
        docs,
        key=lambda x: x.formation_energy_per_atom
        if x.formation_energy_per_atom is not None
        else 1e9
    )

    structure = best_doc.structure
    cif_text = structure.to(fmt="cif")

    return {
        "identifier": str(best_doc.material_id),
        "formula": best_doc.formula_pretty,
        "formation_energy_per_atom": best_doc.formation_energy_per_atom,
        "structure": structure,
        "cif_text": cif_text
    }

In [5]:
# =========================
# Cell 5 - Hàm chuyển structure sang Atoms giống ảnh 1
# =========================
def structure_to_ase_atoms_string(structure):
    """
    Chuyển pymatgen Structure sang ASE Atoms string.
    """
    atoms = structure.to_ase_atoms()
    return str(atoms)


from pymatgen.core import Composition

def get_num_atoms_formula(formula):
    comp = Composition(formula)
    return int(sum(comp.get_el_amt_dict().values()))

In [6]:
# =========================
# Cell 6 - Hàm chính: đọc CSV, lọc stoichiometry, lấy CIF từ MP
# =========================
def build_mp_cif_dataframe(
    input_csv,
    api_key,
    output_csv=None,
    cif_folder="downloaded_cif",
    max_rows=None
):
    """
    Đọc file CSV gồm các cột:
        Formula, Temperature (K), TC, DOI

    Trả về DataFrame gồm:
        identifier, formula, atoms, num_atoms,
        Formula_original, Temperature (K), TC, DOI,
        formation_energy_per_atom, cif_path
    """

    os.makedirs(cif_folder, exist_ok=True)

    df = pd.read_csv(input_csv)

    required_cols = ["Formula", "Temperature (K)", "TC", "DOI"]
    for col in required_cols:
        if col not in df.columns:
            raise ValueError(f"Thiếu cột: {col}")

    df = df[required_cols].copy()
    df = df.dropna(subset=["Formula", "Temperature (K)", "TC"])

    # lọc stoichiometry
    df["is_stoichiometry"] = df["Formula"].apply(is_stoichiometric_formula)
    df_stoi = df[df["is_stoichiometry"]].copy()

    if max_rows is not None:
        df_stoi = df_stoi.head(max_rows)

    results = []

    # cache để tránh query MP lặp lại cùng một công thức
    mp_cache = {}

    for _, row in tqdm(df_stoi.iterrows(), total=len(df_stoi)):
        formula_original = row["Formula"]

        try:
            formula_query = normalize_formula(formula_original)
        except Exception:
            continue

        if formula_query not in mp_cache:
            try:
                mp_result = get_lowest_energy_structure_from_mp(
                    formula=formula_query,
                    api_key=api_key
                )
                mp_cache[formula_query] = mp_result
            except Exception as e:
                print(f"Lỗi khi query {formula_query}: {e}")
                mp_cache[formula_query] = None

        mp_result = mp_cache[formula_query]

        if mp_result is None:
            continue

        structure = mp_result["structure"]

        # lưu file CIF
        cif_filename = f"{mp_result['identifier']}_{formula_query}.cif"
        cif_path = os.path.join(cif_folder, cif_filename)

        with open(cif_path, "w", encoding="utf-8") as f:
            f.write(mp_result["cif_text"])

        results.append({
            "identifier": mp_result["identifier"],
            "formula": mp_result["formula"],
            "atoms": structure_to_ase_atoms_string(structure),
            "num_atoms": get_num_atoms_formula(mp_result["formula"]),

            "Formula_original": formula_original,
            "Temperature (K)": row["Temperature (K)"],
            "TC": row["TC"],
            "DOI": row["DOI"],

            "formation_energy_per_atom": mp_result["formation_energy_per_atom"],
            "cif_path": cif_path
        })

    result_df = pd.DataFrame(results)

    if output_csv is not None:
        result_df.to_csv(output_csv, index=False, encoding="utf-8-sig")
        print("Đã lưu:", output_csv)

    return result_df

In [7]:
# =========================
# Cell 7 - Chạy code
# =========================
API_KEY = "Vdjc3ZVzQuDSoF2c5M14BHVWoUV94SS3"

input_csv = "/content/TC_filtered_final.csv"

result_df = build_mp_cif_dataframe(
    input_csv=input_csv,
    api_key=API_KEY,
    output_csv="mp_cif_matched_data.csv",
    cif_folder="downloaded_cif",
    max_rows=None
)

result_df.head()

  1%|          | 1209/99039 [00:36<52:15, 31.20it/s]

Retrieving SummaryDoc documents:   0%|          | 0/1 [00:00<?, ?it/s]

  1%|▏         | 1436/99039 [00:37<05:46, 281.61it/s]

Retrieving SummaryDoc documents:   0%|          | 0/89 [00:00<?, ?it/s]

  2%|▏         | 1536/99039 [00:38<12:27, 130.39it/s]

...

100%|█████████▉| 98963/99039 [21:25<00:00, 177.42it/s]

Retrieving SummaryDoc documents:   0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 99039/99039 [21:26<00:00, 77.01it/s] 


Đã lưu: mp_cif_matched_data.csv


,identifier,formula,atoms,num_atoms,Formula_original,Temperature (K),TC,DOI,formation_energy_per_atom,cif_path
0,mp-1202900,Al13Fe4,"MSONAtoms(symbols='Al39Fe12', pbc=True, cell=[...",17,Al13Fe4,1.60000,3.514180,10.1103/physrevb.81.184203,-0.32675,downloaded_cif/mp-1202900_Al13Fe4.cif
1,mp-1202900,Al13Fe4,"MSONAtoms(symbols='Al39Fe12', pbc=True, cell=[...",17,Al13Fe4,2.13333,0.678175,10.1103/physrevb.81.184203,-0.32675,downloaded_cif/mp-1202900_Al13Fe4.cif
2,mp-1202900,Al13Fe4,"MSONAtoms(symbols='Al39Fe12', pbc=True, cell=[...",17,Al13Fe4,2.40000,4.377310,10.1103/physrevb.81.184203,-0.32675,downloaded_cif/mp-1202900_Al13Fe4.cif
3,mp-1202900,Al13Fe4,"MSONAtoms(symbols='Al39Fe12', pbc=True, cell=[...",17,Al13Fe4,3.46667,5.487050,10.1103/physrevb.81.184203,-0.32675,downloaded_cif/mp-1202900_Al13Fe4.cif
4,mp-1202900,Al13Fe4,"MSONAtoms(symbols='Al39Fe12', pbc=True, cell=[...",17,Al13Fe4,3.73333,1.787920,10.1103/physrevb.81.184203,-0.32675,downloaded_cif/mp-1202900_Al13Fe4.cif
